<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/08%20-%20Sequence%20Models%20and%20Transformers/08a_smiles_language_model.ipynb)

# 08a · 让模型写分子：SMILES 语言模型

> 对应 ai4chem `05 - Generative Models` 的 RNN 部分。原课程用 ZINC 的类药分子和一个下载来的预训练模型；这里换成**有机光伏候选分子**（哈佛清洁能源计划 CEP），并且从分词器开始一步步自己搭。

前四部分都是在**已有的**分子上做预测和分析。这一部分换一个问题：**能不能让模型自己写出新分子？**

1. 生成模型在做什么？
2. SMILES 分词：把分子拆成"字"
3. 二元模型：一个不用神经网络的基线
4. 搭一个 SMILES 语言模型（LSTM）
5. 训练
6. 采样：温度、有效性、唯一性、新颖性
7. 它真的学到化学了吗？

### 这个 notebook 怎么用

| 标记 | 意思 | 你要做什么 |
|---|---|---|
| 📘 | 讲解 | 读一读 |
| 🤔 | 先猜一猜 | **运行之前**先想想会输出什么 |
| ▶️ | 运行看看 | 选中代码格，按 `Shift + Enter` |
| ✏️ | 改一改 | 改代码里标了 `# ✏️` 的地方，再运行 |
| 🏋️ | 练习 | 自己写代码；参考答案在 `solutions/` 文件夹 |
| 💡 | 小贴士 | 有用但不是必须记住的东西 |


💡 这个 notebook 会训练一个神经网络，CPU 上大约 3 分钟。后面的分析用我们提供的预训练模型（`data/cep_rnn.pt`），这样结果更好看，也和 09b 保持一致。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "08 - Sequence Models and Transformers"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


In [ ]:
try:
    import rdkit
except ImportError:
    %pip install -q rdkit
    import rdkit

import os
import re
import collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, Descriptors

RDLogger.DisableLog("rdApp.*")
torch.manual_seed(0)
np.random.seed(0)
print("PyTorch 版本:", torch.__version__)

---
## 1. 生成模型在做什么？

📘 第二、三部分的模型是**判别式**的：给一个分子，输出一个数（logS）。生成模型反过来：**不给输入，输出一个分子**。

听起来很神奇，其实思路和手机输入法的联想词一样：

> 看过很多很多句子以后，给定前面几个字，猜下一个字最可能是什么。

把"句子"换成 SMILES 字符串，"字"换成原子和键的符号，就得到了一个**分子语言模型**。反复"猜下一个字"，就能一个一个字地写出一个全新的分子。

⚠️ 先说清楚一件事，这是整个部分最重要的一句话：

> **训练一个分子生成模型，不是教它"生成分子"，而是教它复现训练数据的概率分布。**

它写出来的分子会像训练集里的分子。训练数据是药物，它就写药物；训练数据是有机光伏分子，它就写有机光伏分子。模型不会凭空产生化学直觉——**数据才是化学**。第 7 节我们会亲自验证这一点。

### 1.1 这一部分的数据：有机光伏候选分子

📘 前四部分用的是 AqSolDB（水溶解度）。这一部分换成 **CEP**（Harvard Clean Energy Project）：约 3 万个有机光伏（OPV）候选分子，每个都标注了用 Scharber 模型从 DFT 的 HOMO/LUMO 估算出来的**能量转换效率 PCE**（%）。

换数据的原因有两个：

1. 这些分子**正是这门课的主题**：噻吩、呋喃、硒吩、苯并噻二唑、咔唑、硅杂环戊二烯……全是有机半导体的构筑单元；
2. 它们的结构**高度同质**（都是稠合共轭环系），语言模型比较容易学会，CPU 上几分钟就能训练出像样的结果。

💡 PCE 是**计算出来的**估计值，不是实验测量值。真实的器件效率还取决于形貌、界面、加工工艺等等，Scharber 模型只是一个粗略的上限估计。所以这里的 PCE 应当理解成"一个和分子电子结构有关的、可以优化的数字"，而不是真实效率。

In [ ]:
import os
import pandas as pd

# 本地运行：数据在 data/ 文件夹里
# Colab 运行：如果找不到文件，会弹出上传按钮，选择 cep.csv 即可
DATA_DIR = "data"
if not os.path.exists(os.path.join(DATA_DIR, "cep.csv")):
    DATA_DIR = "."
if not os.path.exists(os.path.join(DATA_DIR, "cep.csv")):
    try:
        from google.colab import files
        print("请上传 cep.csv（如果要用预训练模型，把 cep_rnn.pt 也一起上传）")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 cep.csv，请把它放在 data/ 文件夹或和 notebook 同一个文件夹里")

df = pd.read_csv(os.path.join(DATA_DIR, "cep.csv"))
print("读取成功！一共", len(df), "个分子")
df.head()

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]
print("能被 RDKit 解析的分子:", sum(m is not None for m in mols), "/", len(df))
print("重原子数: 平均", np.mean([m.GetNumHeavyAtoms() for m in mols]).round(1),
      " 范围", min(m.GetNumHeavyAtoms() for m in mols), "-", max(m.GetNumHeavyAtoms() for m in mols))
print("SMILES 长度: 平均", df["smiles"].str.len().mean().round(1), " 最长", df["smiles"].str.len().max())

Draw.MolsToGridImage(mols[:8], molsPerRow=4, subImgSize=(260, 200),
                     legends=[f"PCE {p:.1f}%" for p in df["PCE"][:8]])

🤔 看看这些分子的 SMILES，和第一部分的 AqSolDB 比，它们有什么特点？

In [ ]:
for s in df["smiles"][:5]:
    print(s)

📘 全是小写字母（芳香原子）、数字（成环）和少量 `[nH]`、`[SiH2]`、`[se]`，几乎没有支链和官能团。这是一批"长得很像"的稠环共轭分子——对语言模型来说是很友好的训练数据。

---
## 2. SMILES 分词

📘 模型不能直接吃字符串，要先把它切成一个个**token**（词元）。最朴素的做法是按单个字符切，但这样会切坏两类东西：

- `Cl`、`Br` 会被切成 `C`+`l`、`B`+`r`，而 `B`（硼）和 `C`（碳）是完全不同的原子；
- `[nH]`、`[SiH2]`、`[se]` 这些方括号整体才有意义。

所以化学里的标准做法是用**正则表达式**：方括号整体算一个 token，`Cl`/`Br` 算一个，其余按单字符切。

In [ ]:
TOKEN_PATTERN = re.compile(r"(\[[^\]]+\]|Br|Cl|[BCNOPSFIbcnops]|\(|\)|\.|=|#|-|\+|\\|/|:|%\d{2}|\d)")

def tokenize(smiles):
    tokens = TOKEN_PATTERN.findall(smiles)
    assert "".join(tokens) == smiles, f"分词结果拼不回原串: {smiles}"   # 一定要检查！
    return tokens

print(tokenize("c1ccsc1"))
print(tokenize("[nH]1c(cc2c3nsnc3c3c4ccccc4oc3c12)-c1ccco1"))
print(tokenize("CC(=O)Oc1ccccc1C(=O)O"))     # 阿司匹林，带支链和官能团

💡 那句 `assert` 很重要：如果正则表达式漏掉了某种写法，分出来的 token 拼不回原串，就会立刻报错，而不是悄悄地训练出一个错误的模型。**对自己的数据处理加一个"能还原"的检查**，是个很好的习惯。

▶️ 看看整个数据集用到了哪些 token：

In [ ]:
token_lists = [tokenize(s) for s in df["smiles"]]
counter = collections.Counter(t for ts in token_lists for t in ts)
print("一共", len(counter), "种 token")
pd.Series(counter).sort_values(ascending=False).to_frame("count")

📘 只有 20 种 token。对比一下：ZINC 那样的类药分子数据集，token 数通常有 30–100 种（还有立体化学、电荷、更多元素）。**词表小 = 模型好学**。

现在加上三个特殊 token，建立"token ↔ 编号"的对照表：

- `<pad>`（编号 0）：补齐用。不同分子长度不同，要补成一样长才能放进一个张量；
- `^`：开始符。采样时模型从它开始；
- `$`：结束符。模型输出它，就表示"这个分子写完了"。

In [ ]:
vocab = ["<pad>", "^", "$"] + sorted(counter)
stoi = {t: i for i, t in enumerate(vocab)}            # token -> 编号
max_len = max(len(ts) for ts in token_lists) + 2      # +2 是给 ^ 和 $ 留位置
print("词表大小:", len(vocab), " 最长序列:", max_len)
print(vocab)

data = torch.zeros(len(token_lists), max_len, dtype=torch.long)
for i, tokens in enumerate(token_lists):
    ids = [stoi["^"]] + [stoi[t] for t in tokens] + [stoi["$"]]
    data[i, :len(ids)] = torch.tensor(ids)

print("\n第一个分子:", df["smiles"][0])
print("编码后（0 是补齐）:", data[0][:20].tolist(), "...")

---
## 3. 二元模型：一个不用神经网络的基线

📘 第二部分讲过，**任何模型都要和一个笨办法比一比**（02a 的"永远猜平均值"）。生成模型的笨办法叫 **bigram（二元模型）**：只看前**一个** token，统计训练数据里它后面最常跟什么，然后按这个频率抽签。

🤔 只看前一个 token，能写出合法的分子吗？先猜一猜：生成 1000 个，有多少能被 RDKit 读懂？

In [ ]:
bigram = collections.defaultdict(collections.Counter)
for tokens in token_lists:
    seq = ["^"] + tokens + ["$"]
    for a, b in zip(seq, seq[1:]):
        bigram[a][b] += 1

rng = np.random.RandomState(0)

def bigram_sample(max_steps=max_len):
    out, current = [], "^"
    for _ in range(max_steps):
        options = list(bigram[current])
        probs = np.array([bigram[current][o] for o in options], dtype=float)
        current = options[rng.choice(len(options), p=probs / probs.sum())]
        if current == "$":
            break
        out.append(current)
    return "".join(out)

for _ in range(5):
    print(bigram_sample())

In [ ]:
canonical_train = {Chem.CanonSmiles(s) for s in df["smiles"]}

def evaluate(samples, name="model"):
    """生成模型的三个标准指标：有效性、唯一性、新颖性。"""
    mols = [Chem.MolFromSmiles(s) for s in samples]
    valid = [Chem.MolToSmiles(m) for m in mols if m is not None and m.GetNumAtoms() > 0]
    unique = set(valid)
    novel = unique - canonical_train
    row = {
        "valid": len(valid) / len(samples),                 # 能被 RDKit 读懂的比例
        "unique": len(unique) / max(len(valid), 1),         # 有效分子里不重复的比例
        "novel": len(novel) / max(len(unique), 1),          # 不重复分子里训练集没有的比例
    }
    print(f"{name:22s} valid {row['valid']:.3f}  unique {row['unique']:.3f}  novel {row['novel']:.3f}")
    return row

In [ ]:
bigram_samples = [bigram_sample() for _ in range(1000)]
bigram_scores = evaluate(bigram_samples, "bigram")

📘 有效率约 0.3%：一千个里只有两三个能读懂。

🤔 为什么这么差？它的 token 顺序明明都是训练数据里真实出现过的。

📘 因为 SMILES 有**长程约束**，只看前一个字符完全看不到：

- 括号必须配对：`(` 开了就得有 `)`；
- 环的编号必须成对出现：`c1ccc...` 必须在后面某处再出现一个 `1` 把环闭上；
- 化合价要合理。

这些约束跨越几十个 token。要学会它们，模型必须**记住前面写过什么**——这正是下一节 LSTM 要做的事。

---
## 4. 搭一个 SMILES 语言模型

📘 模型要做的事只有一件：**给定前面所有的 token，预测下一个 token 的概率分布。**

$$P(S) = P(s_1) \cdot P(s_2 \mid s_1) \cdot P(s_3 \mid s_1, s_2) \cdots P(s_T \mid s_1, \ldots, s_{T-1})$$

三个部件：

| 部件 | 作用 |
|---|---|
| `nn.Embedding` | 把 token 编号变成向量（10a 的原子特征是手工设计的 one-hot，这里的向量是**学出来的**） |
| `nn.LSTM` | 循环层。每读一个 token 就更新一次**隐状态**，隐状态就是模型对"前面写过什么"的记忆 |
| `nn.Linear` | 把隐状态变成每个 token 的分数（logits），softmax 之后就是概率 |

💡 LSTM（长短期记忆）是 RNN 的一种，专门设计了"门"来决定记住什么、忘记什么，比最朴素的 RNN 更能处理长程依赖。现在的大语言模型用的是 Transformer，但在这种几十个 token 的短序列上，LSTM 又快又好。

In [ ]:
class SmilesRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_size=256, n_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)   # 每个 token -> 一个向量
        self.lstm = nn.LSTM(embedding_dim, hidden_size, n_layers, batch_first=True, dropout=0.1)
        self.linear = nn.Linear(hidden_size, vocab_size)           # 隐状态 -> 每个 token 的分数

    def forward(self, x, state=None):
        """x: (batch, 长度) 的 token 编号。返回每个位置上"下一个 token"的分数。"""
        out, state = self.lstm(self.embedding(x), state)
        return self.linear(out), state

In [ ]:
model = SmilesRNN(len(vocab))
print(model)
print("参数量:", sum(p.numel() for p in model.parameters()))

### 4.1 训练目标

📘 训练的技巧叫**teacher forcing**：把整条真实序列一次性喂进去，让模型在**每一个位置**都预测下一个 token。

```
输入:  ^  c  1  c  c  s  c  1
目标:  c  1  c  c  s  c  1  $
```

一条长度 50 的 SMILES 就提供了 50 个训练样本。损失函数是交叉熵（07a 练习 07a-3 的二分类交叉熵的多分类版本），`ignore_index=0` 让补齐位置不参与计算。

🤔 一个**完全没训练**的模型，预测会是什么样？

In [ ]:
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

example = data[:1]
with torch.no_grad():
    logits, _ = model(example[:, :-1])
    probs = torch.softmax(logits[0, 0], dim=-1)     # 看到 ^ 之后，下一个 token 的概率
print("未训练模型在 ^ 之后的 token 概率（前 5）:")
print(pd.Series(probs.numpy(), index=vocab).sort_values(ascending=False).head(5).round(4))
print("\n均匀分布应该是 1/%d = %.4f" % (len(vocab), 1 / len(vocab)))
print("初始损失:", round(loss_fn(logits.reshape(-1, len(vocab)), example[:, 1:].reshape(-1)).item(), 3),
      " ln(词表大小) =", np.log(len(vocab)).round(3))

📘 每个 token 的概率都接近 1/23，损失接近 $\ln 23 \approx 3.14$：模型什么都不知道，在瞎猜。

---
## 5. 训练

▶️ 训练 3 轮，CPU 上大约 3 分钟。每轮结束后采样 200 个看看进展：

In [ ]:
@torch.no_grad()
def sample(model, n=500, temperature=1.0, max_steps=max_len):
    """从模型里采样 n 个 SMILES 字符串。"""
    model.eval()
    x = torch.full((n, 1), stoi["^"])             # 所有序列都从 ^ 开始
    state = None
    out = torch.zeros(n, 0, dtype=torch.long)
    finished = torch.zeros(n, dtype=torch.bool)
    for _ in range(max_steps):
        logits, state = model(x, state)            # 只喂最新的一个 token，state 记住了前面的内容
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        x = torch.multinomial(probs, 1)            # 按概率抽一个 token（不是取最大的！）
        out = torch.cat([out, x], dim=1)
        finished |= (x[:, 0] == stoi["$"])
        if finished.all():
            break
    model.train()

    smiles_list = []
    for row in out.tolist():
        tokens = []
        for t in row:
            if t == stoi["$"]:
                break
            tokens.append(vocab[t])
        smiles_list.append("".join(tokens))
    return smiles_list

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_epochs(model, optimizer, rows, n_epochs, lr=1e-3, batch_size=128):
    for group in optimizer.param_groups:
        group["lr"] = lr
    subset = data[rows]
    for epoch in range(n_epochs):
        perm = np.random.permutation(len(subset))
        total = 0.0
        for i in range(0, len(perm), batch_size):
            batch = subset[perm[i:i + batch_size]]
            logits, _ = model(batch[:, :-1])                                  # 预测
            loss = loss_fn(logits.reshape(-1, len(vocab)), batch[:, 1:].reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5)                   # 防止梯度爆炸
            optimizer.step()
            total += loss.item()
        samples = sample(model, 200)
        valid = sum(Chem.MolFromSmiles(s) is not None for s in samples) / 200
        print(f"epoch {epoch}: loss {total / (len(perm) / batch_size):.3f}  有效率 {valid:.2f}")

train_epochs(model, optimizer, np.arange(len(data)), n_epochs=3)

📘 损失从 3.1 掉到 0.4 左右，有效率从 0 涨到 30%–60%。模型正在学会括号配对和闭环。

💡 `clip_grad_norm_` 把梯度的长度限制在 5 以内。RNN 在长序列上容易出现**梯度爆炸**（梯度一层层乘起来变得极大，一步就把参数毁掉），裁剪是标准做法。

### 5.1 换上训练好的模型

📘 3 轮还不够。我们提供了一个训练 20 轮的模型，下面的分析都用它。

In [ ]:
CKPT = os.path.join(DATA_DIR, "cep_rnn.pt")
rnn = SmilesRNN(len(vocab))
if os.path.exists(CKPT):
    rnn.load_state_dict(torch.load(CKPT, map_location="cpu"))
    print("已加载预训练模型:", CKPT)
else:
    print("找不到 cep_rnn.pt，继续用刚才训练 3 轮的模型（结果会差一些）")
    rnn = model
rnn.eval();

---
## 6. 采样

📘 采样就是反复做这件事：喂进当前 token → 得到下一个 token 的概率 → **按概率抽签** → 把抽到的 token 喂回去，直到抽到 `$`。

🤔 为什么要"抽签"，不直接每次都取概率最大的那个 token？

📘 因为那样每次都会得到**完全相同**的分子（贪心解码是确定的）。生成模型的价值在于多样性，所以要随机采样。

代码在上面的 `sample` 函数里，有两个细节值得注意：

- `model(x, state)` 每次只喂**一个**新 token，`state` 带着前面所有的记忆。这比每次都把整条序列重算一遍快得多；
- `temperature`（温度）用来调节随机性，下面马上讲。

In [ ]:
samples = sample(rnn, 1000)
print("前 5 个生成的 SMILES:")
for s in samples[:5]:
    print(" ", s)
rnn_scores = evaluate(samples, "LSTM (T=1.0)")

📘 三个指标是生成模型的标准报告项：

| 指标 | 含义 | 不好的情况 |
|---|---|---|
| **valid**（有效性） | 能被 RDKit 读懂 | 低 = 没学会语法 |
| **unique**（唯一性） | 有效分子里不重复的比例 | 低 = **模式崩溃**，翻来覆去写同几个分子 |
| **novel**（新颖性） | 训练集里没有的比例 | 低 = 在**背诵**训练数据 |

三个都高，才说明模型真的学到了东西而不是在作弊。

▶️ 画出来看看：

In [ ]:
valid_mols = [m for m in (Chem.MolFromSmiles(s) for s in samples) if m is not None and m.GetNumAtoms() > 0]
Draw.MolsToGridImage(valid_mols[:12], molsPerRow=4, subImgSize=(260, 200))

🤔 和第 1 节训练数据的图比一比。像不像同一批人画出来的？

### 6.1 温度

📘 **温度** $T$ 在 softmax 之前除一下 logits：$p_i \propto \exp(z_i / T)$。

- $T < 1$：概率分布变"尖"，模型更保守，只走它最有把握的路；
- $T = 1$：按模型学到的真实分布采样；
- $T > 1$：分布变"平"，更冒险、更多样，但也更容易出错。

🤔 温度升高，有效性和唯一性会怎么变？

In [ ]:
rows = []
for T in [0.5, 0.7, 1.0, 1.2, 1.5]:                 # ✏️ 试试别的温度
    scores = evaluate(sample(rnn, 1000, temperature=T), f"T = {T}")
    rows.append({"T": T, **scores})
temperature_table = pd.DataFrame(rows).set_index("T")

fig, ax = plt.subplots(figsize=(5.5, 3.5))
temperature_table.plot(marker="o", ax=ax)
ax.set_xlabel("temperature"); ax.set_ylabel("fraction"); ax.set_ylim(0, 1.05)
ax.set_title("Effect of sampling temperature")
plt.tight_layout(); plt.show()

📘 典型的权衡：**温度越低越"安全"，有效率高，但分子越来越千篇一律；温度越高越多样，但废品越多。**

实际用的时候，常见做法是用稍低的温度（0.7–1.0）多采样一些，再把无效的扔掉。

### 🏋️ 练习 08a-1

模型写出来的无效 SMILES，错在哪里？写代码把 `sample(rnn, 2000)` 里无效的 SMILES 分成两类：

1. **语法错**：`Chem.MolFromSmiles(s, sanitize=False)` 就返回 `None`（括号或环号不配对，RDKit 连读都读不出来）；
2. **化学错**：能读出来，但 `Chem.SanitizeMol` 过不了（化合价不对、芳香环不能 kekulize 等）。

统计两类各占多少，并各打印 3 个例子。提示：用 `Chem.DetectChemistryProblems(m)` 可以拿到第二类的具体问题类型。

In [ ]:
# 在这里写你的代码

---
## 7. 它真的学到化学了吗？

📘 回到第 1 节那句话：**模型学的是训练数据的分布**。现在来验证。

### 7.1 性质分布

▶️ 比较训练集和生成分子的几个描述符分布：

In [ ]:
generated = sample(rnn, 2000, temperature=1.0)
gen_mols = [m for m in (Chem.MolFromSmiles(s) for s in generated) if m is not None and m.GetNumAtoms() > 0]
train_mols = [mols[i] for i in np.random.RandomState(0).choice(len(mols), 2000, replace=False)]

properties = {
    "heavy atoms": lambda m: m.GetNumHeavyAtoms(),
    "aromatic rings": Descriptors.NumAromaticRings,
    "MolLogP": Descriptors.MolLogP,
    "TPSA": Descriptors.TPSA,
}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, (name, fn) in zip(axes, properties.items()):
    ax.hist([fn(m) for m in train_mols], bins=25, density=True, alpha=0.6, label="CEP training data")
    ax.hist([fn(m) for m in gen_mols], bins=25, density=True, alpha=0.6, label="generated")
    ax.set_title(name)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

📘 两条分布基本重合。模型没有被告知"请生成 25–30 个重原子、3–5 个芳香环的分子"，它只是**模仿**了训练数据。

### 7.2 它用的是哪些结构单元？

▶️ 统计几个有机半导体常见构筑单元在训练集和生成分子里的出现频率：

In [ ]:
fragments = {
    "thiophene": "c1ccsc1",
    "furan": "c1ccoc1",
    "pyrrole": "c1cc[nH]c1",
    "selenophene": "c1cc[se]c1",
    "thiadiazole": "c1nsnc1",
    "benzene": "c1ccccc1",
}
rows = []
for name, smarts in fragments.items():
    patt = Chem.MolFromSmiles(smarts)
    rows.append({
        "fragment": name,
        "CEP training": np.mean([m.HasSubstructMatch(patt) for m in train_mols]),
        "generated": np.mean([m.HasSubstructMatch(patt) for m in gen_mols]),
    })
fragment_table = pd.DataFrame(rows).set_index("fragment")
fragment_table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.5))
fragment_table.plot.barh(ax=ax)
ax.set_xlabel("fraction of molecules containing the fragment")
ax.set_title("Building blocks: training data vs generated")
plt.tight_layout(); plt.show()

📘 各个构筑单元的出现频率几乎一样。**模型学会的是"CEP 分子长什么样"这件事的统计规律**，而不是任何关于光伏的物理。

🤔 那么问题来了：如果我们想要**高效率**的分子，而不只是"看起来像 CEP"的分子，该怎么办？

📘 这正是 **09b** 的内容：把生成模型和第二部分的性质预测模型结合起来，做**有目标的生成**。

### 🏋️ 练习 08a-2

生成模型是在**背诵**吗？第 6 节的 novel 指标说 98% 以上的生成分子不在训练集里，但"不完全相同"不等于"不像"。

对 500 个生成分子，计算每个分子到**训练集**的最大 Tanimoto 相似度（Morgan 指纹，半径 2，2048 位；03b 学过），画出分布。再和一个对照组比较：从训练集里随机抽 500 个分子，算它们到**其余训练分子**的最大相似度。

两条分布像不像？这说明生成分子和训练分子的"距离"，与训练分子彼此之间的距离相比如何？

提示：`DataStructs.BulkTanimotoSimilarity(fp, fp_list)` 一次算一个对多个。训练集全算太慢，随机抽 3000 个做参照即可。

### 🏋️ 练习 08a-3

把 `sample` 改成**贪心解码**（每一步都取概率最大的 token，而不是抽签），生成 10 个分子。它们一样吗？结合第 6 节开头的问题解释你看到的结果。

提示：把 `torch.multinomial(probs, 1)` 换成 `probs.argmax(dim=-1, keepdim=True)`。

In [ ]:
# 在这里写你的代码

---
## 小结

| 学到了 | 代码 / 要点 |
|---|---|
| 生成模型的本质 | 学习训练数据的概率分布，不是"创造" |
| SMILES 分词 | 正则表达式；方括号整体、`Cl`/`Br` 不拆；加 `assert` 检查能还原 |
| 特殊 token | `<pad>` 补齐、`^` 开始、`$` 结束 |
| 基线 | bigram 有效率只有 0.3%：长程约束需要记忆 |
| 模型 | `nn.Embedding` + `nn.LSTM` + `nn.Linear` |
| 训练 | teacher forcing；交叉熵；`clip_grad_norm_` |
| 采样 | 按概率抽签（不是取最大）；用 `state` 增量推进 |
| 温度 | 低 = 安全但单调，高 = 多样但易错 |
| 三个指标 | valid / unique / novel |

**下一节 09a**：LSTM 一个字一个字地写分子，中间没有一个"分子的向量表示"。**变分自编码器（VAE）** 会给每个分子一个连续的向量坐标，于是我们可以在分子之间做插值、在"化学空间"里导航。